In [83]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import numpy as np
from gensim.models import Word2Vec

In [ ]:

splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

df_train = df_train[df_train['lang'].isin(['ar', 'ko', 'te'])]
df_val = df_val[df_val['lang'].isin(['ar', 'ko', 'te'])]


df_train.head()


,question,context,lang,answerable,answer_start,answer,answer_inlang
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None
4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None


In [ ]:
def cleanDf(df):
    pattern = re.compile(r"[?؟,;\/\\\[\]#():]")
    # pattern_context = re.compile(r"[?؟,;\/\\\[\]#():.]")
    df['question'] = df['question'].apply(lambda x: pattern.sub("", x))
    df['context'] = df['context'].apply(lambda x: pattern.sub("", x))
    return df

def word_overlap(context, question):

    context_words = set(context.lower().split())
    question_words = set(question.lower().split())

    overlap = context_words.intersection(question_words)
    
    if len(question_words) == 0:
        return 0.0
    return len(overlap) / len(question_words)

def train_word2vec(df, embedding_dim=100):
    sentences = []
    for _, row in df.iterrows():

        sentences.append(row['context'].lower().split())
        sentences.append(row['question'].lower().split())
    
    model = Word2Vec(sentences, vector_size=100, window=10, min_count=1, sg=1, epochs=30)

    return model

def avg_embedding(tokens, wv, dim=100):
    vecs = [wv[t] for t in tokens if t in wv]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)


def cosine_similarity(vec1, vec2):
    if np.linalg.norm(vec1) == 0 or np.linalg.norm(vec2) == 0:
        return 0.0
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def build_features(df, w2v_model, embedding_dim=100):
    X = []
    for c, q in zip(df['context'], df['question']):
        overlap = word_overlap(c, q)
        c_tokens = c.lower().split()
        q_tokens = q.lower().split()
        c_vec = avg_embedding(c_tokens, w2v_model.wv, embedding_dim)
        q_vec = avg_embedding(q_tokens, w2v_model.wv, embedding_dim)
        cos_sim = cosine_similarity(c_vec, q_vec)
        X.append([overlap, cos_sim])
    return torch.tensor(X, dtype=torch.float32)


In [86]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self,x):
        return self.linear(x)

In [ ]:
df_train = cleanDf(df_train)
df_val = cleanDf(df_val)

languages = ['ar', 'ko', 'te']

results = {}


embedding_dim = 100

def build_text(question, context): 
    return "[CLS]" + question + "[SEP]" + context + "[SEP]"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



for lang in languages:
    print(f"Training and evaluating for language: {lang.upper()}")

    df_train_lang = df_train[df_train['lang'] == lang]
    df_val_lang = df_val[df_val['lang'] == lang]
    w2v_model = train_word2vec(df_train_lang, embedding_dim=embedding_dim)
    
    X_train_tensor = build_features(df_train_lang, w2v_model).to(device)
    X_val_tensor = build_features(df_val_lang, w2v_model).to(device)
    
    y_train_tensor = torch.tensor(df_train_lang['answerable'].values, dtype=torch.float32, device=device).view(-1, 1)
    y_val_tensor = torch.tensor(df_val_lang['answerable'].values, dtype=torch.float32, device=device).view(-1, 1)

    input_dim = X_train_tensor.shape[1]
    model = LogisticRegressionModel(input_dim=input_dim).to(device)

    num_positive = (y_train_tensor == 1).sum().item()
    num_negative = (y_train_tensor == 0).sum().item()

    pos_weight = torch.tensor(num_negative / num_positive, dtype=torch.float32, device = device)


    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    num_epochs = 100

    for epoch in range(num_epochs):

        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

        
    with torch.no_grad():
        logits = model(X_val_tensor)
        y_pred_class = (torch.sigmoid(logits) >= 0.5).float()

        y_true = y_val_tensor.cpu().numpy()
        y_pred_np = y_pred_class.cpu().numpy()

        report = classification_report(y_true, y_pred_np, target_names=['Impossible', 'Answerable'])
        print(f"Classification Report for {lang.upper()}:\n{report}")


    

Training and evaluating for language: AR
Epoch 10/100, Loss: 0.1410
Epoch 20/100, Loss: 0.1393
Epoch 30/100, Loss: 0.1385
Epoch 40/100, Loss: 0.1381
Epoch 50/100, Loss: 0.1380
Epoch 60/100, Loss: 0.1380
Epoch 70/100, Loss: 0.1380
Epoch 80/100, Loss: 0.1380
Epoch 90/100, Loss: 0.1379
Epoch 100/100, Loss: 0.1379
Classification Report for AR:
              precision    recall  f1-score   support

  Impossible       0.16      0.73      0.26        52
  Answerable       0.92      0.43      0.59       363

    accuracy                           0.47       415
   macro avg       0.54      0.58      0.42       415
weighted avg       0.82      0.47      0.55       415

Training and evaluating for language: KO
Epoch 10/100, Loss: 0.0359
Epoch 20/100, Loss: 0.0359
Epoch 30/100, Loss: 0.0358
Epoch 40/100, Loss: 0.0358
Epoch 50/100, Loss: 0.0357
Epoch 60/100, Loss: 0.0357
Epoch 70/100, Loss: 0.0356
Epoch 80/100, Loss: 0.0356
Epoch 90/100, Loss: 0.0356
Epoch 100/100, Loss: 0.0355
Classification Repo

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW  
from sklearn.metrics import classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"

languages = ['ar', 'ko', 'te']
model_name = "xlm-roberta-base"

models = {
    'ar': AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device),
    'ko': AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device),
    'te': AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
}

tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode_examples(df, tokenizer, max_length=256):
    inputs = tokenizer(
        list(df["question"]),
        list(df["context"]),
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    labels = torch.tensor(df["answerable"].values, dtype=torch.long)
    return inputs, labels

for lang in languages:
    print(f"\nTraining model for language: {lang.upper()}")
    
    df_train_lang = df_train[df_train['lang'] == lang]
    df_val_lang = df_val[df_val['lang'] == lang]
    
    train_inputs, train_labels = encode_examples(df_train_lang, tokenizer)
    train_dataset = TensorDataset(train_inputs['input_ids'], train_inputs['attention_mask'], train_labels)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    
    val_inputs, val_labels = encode_examples(df_val_lang, tokenizer)
    val_input_ids = val_inputs['input_ids'].to(device)
    val_attention_mask = val_inputs['attention_mask'].to(device)
    
    model = models[lang]
    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(3):
        for batch in train_loader:
            input_ids, attention_mask, y = [b.to(device) for b in batch]
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        
        print(f"Epoch {epoch+1} done")
    
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=val_input_ids, attention_mask=val_attention_mask)
        y_pred = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        y_true = val_labels.numpy()
    
    print(f"Classification Report for {lang.upper()}:\n")
    print(classification_report(y_true, y_pred, target_names=['Impossible', 'Answerable']))


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

c:\Users\CoolD\miniconda3\envs\NLP\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\CoolD\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

KeyboardInterrupt: 